In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np

##### Load CSV Files #####


# display
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 200)

# data folder
DATA_DIR = '/content/drive/My Drive/Hex525_Data_Science_Project/Data'
#os.listdir(DATA_DIR)

# file names
TEMP_CSV = os.path.join(DATA_DIR, 'temperature_nsw.csv')
DEMAND_CSV = os.path.join(DATA_DIR, 'totaldemand_nsw.csv')
FORECAST_CSV = os.path.join(DATA_DIR, 'forecastdemand_nsw.csv')

# hot/cold theresholds
THRESH_HOT = 30.0   # for slot >= 30°C -> hot slot, daily max >= 30°C -> hot day
THRESH_COLD = 5.0   # for slot <= 5 °C -> cold slot, day min <= 5 -> cold day


In [ ]:

##### Temperature - round to nearest 30 mins, average within slot, build continuous grid #####

df_temp_raw = pd.read_csv(TEMP_CSV)

print("Raw temp shape:", df_temp_raw.shape)
print(df_temp_raw.head(3))

# explicit datetime parse (AU dd/mm/yyyy hh:mm)
df_temp_raw["dt"] = pd.to_datetime(
    df_temp_raw["DATETIME"],
    format="%d/%m/%Y %H:%M",
    utc=True,
    errors="coerce",
)
print("Datetime nulls (temp):", df_temp_raw["dt"].isna().sum())
print("Datetime range (temp):", df_temp_raw["dt"].min(), "to", df_temp_raw["dt"].max())

# round to nearest 30-min slot
df_temp_raw["dt_round"] = df_temp_raw["dt"].dt.round("30min")
print("\nUnique rounded slots sample (temp):", df_temp_raw["dt_round"].sort_values().head(5).tolist())

# average if multiple observations fall in the same rounded slot
df_temp = (
    df_temp_raw
    .groupby("dt_round", as_index=True)["TEMPERATURE"]
    .mean()
    .to_frame("temp")
    .sort_index()
)

# build the continuous 30-min index
full_index = pd.date_range(df_temp.index.min(), df_temp.index.max(), freq="30min")

# align to the grid
df_temp = df_temp.reindex(full_index)

print("Temperature gaps before fill:", df_temp["temp"].isna().sum())

# fill small holes by time interpolation, then pad edges
df_temp["temp"] = (
    df_temp["temp"]
      .interpolate(method="time", limit=6)   # fill gaps up to 3 hours
      .ffill()
      .bfill()
)

print("Temperature gaps after  fill:", df_temp["temp"].isna().sum())

print("\nAfter grouping:", df_temp.shape)
print(df_temp.head(3))
print(df_temp.tail(3))



In [ ]:
df_demand_raw = pd.read_csv(DEMAND_CSV)

print("Raw demand shape:", df_demand_raw.shape)
print(df_demand_raw.head(3))

df_demand_raw["dt"] = pd.to_datetime(
    df_demand_raw["DATETIME"],
    format="%d/%m/%Y %H:%M",
    utc=True,
    errors="coerce",
)
print("Datetime nulls (demand):", df_demand_raw["dt"].isna().sum())
print("Datetime range (demand):", df_demand_raw["dt"].min(), "to", df_demand_raw["dt"].max())


# round to nearest 30-min
df_demand_raw["dt_round"] = df_demand_raw["dt"].dt.round("30min")
print("Unique rounded slots sample (demand):", df_demand_raw["dt_round"].sort_values().head(5).tolist())

# aggregate within slot if duplicates exist
df_demand = (
    df_demand_raw
      .groupby("dt_round", as_index=True)["TOTALDEMAND"]
      .mean()
      .to_frame("y")            # rename to y (target)
      .sort_index()
)

print("\nAfter grouping (demand):", df_demand.shape)
print(df_demand.head(3))
print(df_demand.tail(3))

# updated code for operator forecasts - y_forecast_24_latest

In [ ]:
##### Operator forecasts — “24h-latest” #####

df_forecast_raw = pd.read_csv(FORECAST_CSV)
print("Raw forecast shape:", df_forecast_raw.shape)
print(df_forecast_raw.head(3))

df_forecast_raw["dt_target"] = pd.to_datetime(
    df_forecast_raw["DATETIME"], format="%Y-%m-%d %H:%M:%S", utc=True, errors="raise"
)
df_forecast_raw["dt_issue"] = pd.to_datetime(
    df_forecast_raw["LASTCHANGED"], format="%Y-%m-%d %H:%M:%S", utc=True, errors="raise"
)

print("Datetime nulls (target):", df_forecast_raw["dt_target"].isna().sum())
print("Datetime nulls (issue) :", df_forecast_raw["dt_issue"].isna().sum())
print("Datetime range (target):", df_forecast_raw["dt_target"].min(), "→", df_forecast_raw["dt_target"].max())

# drop unusable rows
before = len(df_forecast_raw)
df_forecast_raw = df_forecast_raw.dropna(subset=["dt_target"]).copy()
after = len(df_forecast_raw)
print(f"Dropped rows with invalid dt_target: {before - after} (kept {after})")

# round target to 30-min grid
df_forecast_raw["dt_round"]       = df_forecast_raw["dt_target"].dt.round("30min")
df_forecast_raw["dt_issue_round"] = df_forecast_raw["dt_issue"].dt.floor("30min")

# latest-issued per target slot (no look-ahead restriction)
df_forecast_latest = (
    df_forecast_raw
      .sort_values(["dt_round", "dt_issue_round"])
      .groupby("dt_round", as_index=False)
      .tail(1)[["dt_round", "FORECASTDEMAND"]]
      .rename(columns={"FORECASTDEMAND": "y_forecast_latest"})
)

# strict 24h-ahead
cut_24h = df_forecast_raw["dt_round"] - pd.Timedelta(hours=24)
df_forecast_24h = (
    df_forecast_raw
      .loc[df_forecast_raw["dt_issue_round"].eq(cut_24h)]
      .sort_values(["dt_round", "dt_issue_round"])
      .groupby("dt_round", as_index=False)
      .tail(1)[["dt_round", "FORECASTDEMAND"]]
      .rename(columns={"FORECASTDEMAND": "y_forecast_24h"})
)

# 24h-latest (fallback)
win_mask = (
    (df_forecast_raw["dt_issue_round"] <= df_forecast_raw["dt_round"]) &
    (df_forecast_raw["dt_issue_round"] >  df_forecast_raw["dt_round"] - pd.Timedelta(hours=24))
)
df_forecast_24h_fallback = (
    df_forecast_raw.loc[win_mask, ["dt_round", "dt_issue_round", "FORECASTDEMAND"]]
      .sort_values(["dt_round", "dt_issue_round"])
      .groupby("dt_round", as_index=False)
      .head(1)
      .rename(columns={"FORECASTDEMAND": "y_forecast_24h_fallback"})
)

# combine strict & fallback -> y_forecast_24h_latest
df_forecast_24h_latest = (
    df_forecast_24h_fallback
      .merge(df_forecast_24h, on="dt_round", how="left")
)
df_forecast_24h_latest["y_forecast_24h_latest"] = (
    df_forecast_24h_latest["y_forecast_24h"]  # prefer strict
      .combine_first(df_forecast_24h_latest["y_forecast_24h_fallback"])
)
df_forecast_24h_latest = df_forecast_24h_latest[["dt_round", "y_forecast_24h_latest"]]

print("Rows (latest)     :", len(df_forecast_latest))
print("Rows (24h strict) :", len(df_forecast_24h))
print("Rows (24h-latest) :", len(df_forecast_24h_latest))
print()
print("Missing (latest / 24h / 24h-latest):",
      df_forecast_latest["y_forecast_latest"].isna().sum(),
      df_forecast_24h["y_forecast_24h"].isna().sum() if len(df_forecast_24h) else 0,
      df_forecast_24h_latest["y_forecast_24h_latest"].isna().sum())
print()
print("dt_round coverage:", df_forecast_latest["dt_round"].min(), "→", df_forecast_latest["dt_round"].max())

# cont. data-prep

In [ ]:
##### Join the three sources on the full 30-min index #####

df_temp      = df_temp.reindex(full_index)
df_demand    = df_demand.reindex(full_index)

# latest-issue forecast -> y_forecast_latest on full index
df_fc_latest = (
    df_forecast_latest
      .set_index("dt_round")[["y_forecast_latest"]]
      .reindex(full_index)
)

# strict 24h-ahead forecast -> y_forecast_24h on full index
df_fc_24h = (
    df_forecast_24h
      .set_index("dt_round")[["y_forecast_24h"]]
      .reindex(full_index)
)

df_fc_24h_latest = (
    df_forecast_24h_latest
      .set_index("dt_round")[["y_forecast_24h_latest"]]
      .reindex(full_index)
)

# merge to one frame
df_all = (
    df_demand.join(df_temp, how="left")
             .join(df_fc_latest, how="left")
             .join(df_fc_24h, how="left")
             .join(df_fc_24h_latest, how="left")

)

df_all = df_all[["temp", "y", "y_forecast_latest", "y_forecast_24h", "y_forecast_24h_latest"]]

print("Joined shape:", df_all.shape)
print(df_all.head(3))
print("Total NaNs by column:\n", df_all.isna().sum())

n_latest      = df_all["y_forecast_latest"].notna().sum()
n_24h_strict  = df_all["y_forecast_24h"].notna().sum()
n_24h_latest  = df_all["y_forecast_24h_latest"].notna().sum()
both_strict   = (df_all["y_forecast_latest"].notna() & df_all["y_forecast_24h"].notna()).sum()
both_24latest = (df_all["y_forecast_latest"].notna() & df_all["y_forecast_24h_latest"].notna()).sum()

print(f"Non-null counts  latest={n_latest:,}  24h_strict={n_24h_strict:,}  24h_latest={n_24h_latest:,}")
print(f"Overlap (latest ∩ 24h_strict)={both_strict:,}  (latest ∩ 24h_latest)={both_24latest:,}")
print("Date range:", df_all.index.min(), "->", df_all.index.max())

In [ ]:
#### testing the values ####

missing_strict = df_all["y_forecast_24h"].isna()

print(f"Total slots with NaN in strict 24h: {missing_strict.sum()}")

filled_by_latest = df_all.loc[missing_strict, "y_forecast_24h_latest"].notna().sum()
print(f"Of these, {filled_by_latest} have a fallback value in y_forecast_24h_latest")

print("\nExamples where strict 24h is NaN but 24h_latest is present:")
print(
    df_all.loc[missing_strict, ["y_forecast_latest", "y_forecast_24h", "y_forecast_24h_latest"]]
         .head(10)
)

In [ ]:
##### Add time/calendar features #####

idx = df_all.index  # datetime index

# hour of day (fractional, e.g. 13.5 = 1:30 pm)
df_all["hour"] = idx.hour + idx.minute/60.0

# day of week (0=Monday, 6=Sunday)
df_all["dow"] = idx.dayofweek

# weekend flag
df_all["is_weekend"] = (df_all["dow"] >= 5).astype(int)

# cyclic encodings for hour of day (48 half-hour slots per day)
df_all["sin_hh"] = np.sin(2*np.pi*(idx.hour*2 + (idx.minute//30))/48)
df_all["cos_hh"] = np.cos(2*np.pi*(idx.hour*2 + (idx.minute//30))/48)

# cyclic encodings for day of week
df_all["sin_dow"] = np.sin(2*np.pi*df_all["dow"]/7)
df_all["cos_dow"] = np.cos(2*np.pi*df_all["dow"]/7)

print(df_all.head(3))


In [ ]:
##### Adding Holiday Feature #####

# NSW Public Holidays 2010–2021
nsw_holidays = pd.to_datetime([
    # 2010
    "2010-01-01","2010-01-26","2010-04-02","2010-04-03","2010-04-04","2010-04-05",
    "2010-04-25","2010-06-14","2010-10-04","2010-12-25","2010-12-26","2010-12-27","2010-12-28",
    # 2011
    "2011-01-01","2011-01-03","2011-01-26","2011-04-22","2011-04-23","2011-04-24","2011-04-25",
    "2011-04-26","2011-06-13","2011-10-03","2011-12-25","2011-12-26","2011-12-27",
    # 2012
    "2012-01-01","2012-01-02","2012-01-26","2012-04-06","2012-04-07","2012-04-08","2012-04-09",
    "2012-04-25","2012-06-11","2012-10-01","2012-12-25","2012-12-26",
    # 2013
    "2013-01-01","2013-01-26","2013-01-28","2013-03-29","2013-03-30","2013-03-31","2013-04-01",
    "2013-04-25","2013-06-10","2013-10-07","2013-12-25","2013-12-26",
    # 2014
    "2014-01-01","2014-01-26","2014-01-27","2014-04-18","2014-04-19","2014-04-20","2014-04-21",
    "2014-04-25","2014-06-09","2014-10-06","2014-12-25","2014-12-26",
    # 2015
    "2015-01-01","2015-01-26","2015-04-03","2015-04-04","2015-04-05","2015-04-06",
    "2015-04-25","2015-06-08","2015-10-05","2015-12-25","2015-12-26","2015-12-28",
    # 2016
    "2016-01-01","2016-01-26","2016-03-25","2016-03-26","2016-03-27","2016-03-28",
    "2016-04-25","2016-06-13","2016-10-03","2016-12-25","2016-12-26","2016-12-27",
    # 2017
    "2017-01-01","2017-01-02","2017-01-26","2017-04-14","2017-04-15","2017-04-16","2017-04-17",
    "2017-04-25","2017-06-12","2017-10-02","2017-12-25","2017-12-26",
    # 2018
    "2018-01-01","2018-01-26","2018-03-30","2018-03-31","2018-04-01","2018-04-02",
    "2018-04-25","2018-06-11","2018-10-01","2018-12-25","2018-12-26",
    # 2019
    "2019-01-01","2019-01-26","2019-01-28","2019-04-19","2019-04-20","2019-04-21","2019-04-22",
    "2019-04-25","2019-06-10","2019-10-07","2019-12-25","2019-12-26",
    # 2020
    "2020-01-01","2020-01-26","2020-01-27","2020-04-10","2020-04-11","2020-04-12","2020-04-13",
    "2020-04-25","2020-06-08","2020-10-05","2020-12-25","2020-12-26","2020-12-28",
    # 2021
    "2021-01-01","2021-01-26","2021-04-02","2021-04-03","2021-04-04","2021-04-05",
    "2021-04-25","2021-06-14","2021-10-04","2021-12-25","2021-12-26","2021-12-27","2021-12-28"
]).date

# add holiday flag
df_all["is_holiday"] = pd.Series(df_all.index.date).isin(nsw_holidays).astype(int).values


df_all["date"] = df_all.index.date


daily = (
    df_all
    .groupby("date", as_index=True)[["is_holiday"]]
    .max()
    .sort_index()
)

# weekend by calendar (0=Mon, ..., 6=Sun)
daily["dow"] = pd.to_datetime(daily.index).weekday
daily["is_weekend"] = (daily["dow"] >= 5).astype(int)


prev_hol = daily["is_holiday"].shift(1).fillna(0)
next_hol = daily["is_holiday"].shift(-1).fillna(0)

daily["is_long_weekend"] = (
    (daily["is_holiday"] == 1) |
    (daily["is_weekend"] & ((prev_hol == 1) | (next_hol == 1)))
).astype(int)

df_all = df_all.join(daily[["is_long_weekend"]], on="date")

print("Days marked long-weekend:", int(daily["is_long_weekend"].sum()))
print(df_all[["is_holiday","is_weekend","is_long_weekend"]].head(10))





In [ ]:
##### Temperature-based flags #####

### Add indicators for extreme conditions. ###

df_all["date"] = df_all.index.date

# slot-level flags
df_all["is_hot_slot"] = (df_all["temp"] >= THRESH_HOT).astype(int)
df_all["is_cold_slot"] = (df_all["temp"] <= THRESH_COLD).astype(int)

# daily max/min
daily_temp = df_all.groupby("date")["temp"].agg(["max", "min"]).sort_index()

# day-level flags
daily_temp["is_hot_day"] = (daily_temp["max"] >= THRESH_HOT).astype(int)
daily_temp["is_cold_day"] = (daily_temp["min"] <= THRESH_COLD).astype(int)
daily_temp["is_normal_day"] = (
    (daily_temp["is_hot_day"] == 0) & (daily_temp["is_cold_day"] == 0)
).astype(int)

# join day-level flags back into df_all
df_all = df_all.join(daily_temp[["is_hot_day", "is_cold_day", "is_normal_day"]], on="date")

print("Days flagged hot (unique days):", int(daily_temp["is_hot_day"].sum()))
print("Days flagged cold (unique days):", int(daily_temp["is_cold_day"].sum()))
print("Days flagged normal (unique days):", int(daily_temp["is_normal_day"].sum()))
print("Slot-level hot flags (records):", int(df_all["is_hot_slot"].sum()))
print("Slot-level cold flags (records):", int(df_all["is_cold_slot"].sum()))
print(df_all.head(5))


In [ ]:
##### Summary counts (day-level) #####

n_days          = len(daily)
n_holidays_d    = int(daily["is_holiday"].sum())
n_weekends_d    = int(daily["is_weekend"].sum())
n_longwk_d      = int(daily["is_long_weekend"].sum())

print(f"Days total:           {n_days}")
print(f"Days that are holidays:       {n_holidays_d}")
print(f"Days that are weekends:       {n_weekends_d}")
print(f"Days that are long-weekend:   {n_longwk_d}")

# show overlaps at day-level
print("\nDay-level cross-tab (holiday x weekend):")
print(pd.crosstab(daily["is_holiday"], daily["is_weekend"], rownames=["holiday"], colnames=["weekend"]))

print("\nDay-level cross-tab (long-weekend x weekend):")
print(pd.crosstab(daily["is_long_weekend"], daily["is_weekend"], rownames=["long_wk"], colnames=["weekend"]))

# summary counts (slot-level)
slot_counts = df_all[["is_holiday","is_weekend","is_long_weekend"]].sum().astype(int)
print("\nSlot-level flagged counts:")
print(slot_counts)

viol = daily.loc[(daily["is_long_weekend"]==1) & ((daily["is_holiday"]==0) & (daily["is_weekend"]==0))]
assert viol.empty, "Found long-weekend days that are neither weekend nor holiday!"
print("\nSanity OK: every long-weekend day is a holiday OR a weekend with an adjacent holiday.")

print("\nExample holiday day:")
print(daily[daily["is_holiday"]==1].head(3))

print("\nExample weekend-only day (not holiday):")
print(daily[(daily["is_weekend"]==1) & (daily["is_holiday"]==0)].head(3))

print("\nExample long-weekend weekend day (weekend with adjacent holiday):")
print(daily[(daily["is_long_weekend"]==1) & (daily["is_weekend"]==1) & (daily["is_holiday"]==0)].head(3))

In [ ]:
##### Lag Features #####

### Lag & rolling features (30-min slots) ###

# demand lags
df_all["y_lag1"]   = df_all["y"].shift(1)
df_all["y_lag48"]  = df_all["y"].shift(48)     # 48 × 30min = 1 day
df_all["y_lag336"] = df_all["y"].shift(336)    # 336 × 30min = 1 week

# rolling demand stats (no leakage: shift(1) first)
df_all["y_roll_mean_48"]  = df_all["y"].shift(1).rolling(48,  min_periods=24).mean()
df_all["y_roll_std_48"]   = df_all["y"].shift(1).rolling(48,  min_periods=24).std()

# recent weekly level (helps seasonality, holidays)
df_all["y_roll_mean_336"] = df_all["y"].shift(1).rolling(336, min_periods=96).mean()

# temperature lags
for k in [1, 2, 3, 48]:                          # 48 = same time yesterday
    df_all[f"temp_lag{k}"] = df_all["temp"].shift(k)

# recent temperature range (captures heating/cooling build-up)
df_all["temp_roll_max_48"] = df_all["temp"].shift(1).rolling(48, min_periods=24).max()
df_all["temp_roll_min_48"] = df_all["temp"].shift(1).rolling(48, min_periods=24).min()

lag_cols = [
    "y_lag1","y_lag48","y_lag336",
    "y_roll_mean_48","y_roll_std_48","y_roll_mean_336",
    "temp_lag1","temp_lag2","temp_lag3","temp_lag48",
    "temp_roll_max_48","temp_roll_min_48"
]
print("Lag NA counts (should be non-zero only at the very beginning):")
print(df_all[lag_cols].isna().sum().sort_values(ascending=False).head(8))

print("\nPreview with lags:")
print(df_all[["y","temp"] + lag_cols].head(50))


In [ ]:
# save to parquet
DATA_DIR = '/content/drive/My Drive/Hex525_Data_Science_Project/Data'

df_all.to_parquet(DATA_DIR+ "/nsw_demand_features2.parquet", index=True)

# save as CSV
df_all.to_csv(DATA_DIR+ "/nsw_demand_features2.csv", index=True)

print("Saved dataset with shape:", df_all.shape)